# ⚔️ Elden Ring Meta-Build Intelligence — Análisis Exploratorio

> **Proyecto Big Data Aplicado 2026** — Gabriel Cardona Medina  
> Notebook de análisis y visualización del pipeline end-to-end.

---

## Objetivo
Demostrar que la pipeline de datos genera conocimiento accionable:
1. **Análisis Batch** (Apache Spark): distribución de efectividad estática
2. **Análisis Streaming** (Apache Flink): evolución del win rate en ventanas de 5 min
3. **Detección de Meta-Shift**: armas cuya popularidad cambió >15% entre ventanas

## 1. Conexión a PostgreSQL (resultado del pipeline)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import psycopg2
from psycopg2.extras import RealDictCursor

# Estilo visual
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'text.color': '#c9d1d9',
    'grid.color': '#21262d',
    'grid.linestyle': '--',
    'grid.linewidth': 0.5,
})
GOLD = '#D4AF37'

conn = psycopg2.connect(
    host='localhost', port=5432,
    database='elden_ring', user='elden', password='ring123'
)
print('✅ Conectado a PostgreSQL')
print(f'   Servidor: PostgreSQL {conn.server_version // 10000}.x')

## 2. Exploración del Modelo de Datos (Batch Layer)

In [ ]:
# Inventario de datos
tables = ['weapons', 'bosses', 'weapon_boss_effectiveness', 'player_events', 'real_time_rankings', 'meta_shifts']
inventory = {}
for t in tables:
    df = pd.read_sql(f'SELECT COUNT(*) AS n FROM {t}', conn)
    inventory[t] = int(df['n'].iloc[0])

print('📊 Inventario de Datos en PostgreSQL:')
print('─' * 40)
for table, count in inventory.items():
    bar = '█' * min(count // 2, 30)
    print(f'  {table:<32} {count:>6,}  {bar}')
print('─' * 40)

In [ ]:
weapons_df = pd.read_sql('SELECT * FROM weapons ORDER BY name', conn)
print(f'Armas: {len(weapons_df)} registros')
weapons_df[['weapon_id','name','category','damage_physical','damage_magic','damage_fire','weight']].head(10)

In [ ]:
bosses_df = pd.read_sql('SELECT * FROM bosses ORDER BY health_points DESC', conn)
print(f'Jefes: {len(bosses_df)} registros')
bosses_df[['boss_id','name','location','health_points','defense_physical']]

## 3. Análisis de Efectividad — Resultado del Job Batch de Spark

In [ ]:
eff_df = pd.read_sql("""
    SELECT w.name AS weapon, b.name AS boss,
           e.effectiveness_score AS score
    FROM weapon_boss_effectiveness e
    JOIN weapons w ON e.weapon_id = w.weapon_id
    JOIN bosses  b ON e.boss_id   = b.boss_id
    ORDER BY w.name, b.name
""", conn)

print(f'Pares de efectividad: {len(eff_df)}')
print(f'Score promedio: {eff_df["score"].mean():.1f}')
print(f'Score máximo: {eff_df["score"].max():.1f} ({eff_df.loc[eff_df["score"].idxmax(), "weapon"]} vs {eff_df.loc[eff_df["score"].idxmax(), "boss"]})')
eff_df.describe()

In [ ]:
# Heatmap de efectividad arma vs jefe
pivot = eff_df.pivot(index='weapon', columns='boss', values='score')

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(
    pivot, annot=True, fmt='.0f', cmap='YlOrRd',
    linewidths=0.5, linecolor='#0d1117',
    annot_kws={'size': 9, 'color': '#0d1117'},
    ax=ax, cbar_kws={'label': 'Score de Efectividad (0-100)'}
)
ax.set_title('⚔️ Matriz de Efectividad Arma × Jefe (Apache Spark Batch)', 
             fontsize=14, pad=15, color=GOLD, fontweight='bold')
ax.set_xlabel('Jefe', fontsize=11)
ax.set_ylabel('Arma', fontsize=11)
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('notebooks/output/heatmap_efectividad.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('📊 Heatmap guardado en notebooks/output/heatmap_efectividad.png')

In [ ]:
# Ranking de armas por efectividad promedio
avg_eff = eff_df.groupby('weapon')['score'].agg(['mean','std','max']).round(1).sort_values('mean', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = [GOLD if i == 0 else '#8b949e' for i in range(len(avg_eff))]
bars = ax.barh(avg_eff.index, avg_eff['mean'], color=colors, edgecolor='none', height=0.7)
ax.errorbar(avg_eff['mean'], range(len(avg_eff)), xerr=avg_eff['std'],
            fmt='none', color='#58a6ff', capsize=4, linewidth=1.5)
for bar, (_, row) in zip(bars, avg_eff.iterrows()):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{row["mean"]:.1f}', va='center', fontsize=9, color='#c9d1d9')
ax.set_title('🏆 Efectividad Promedio por Arma (vs todos los jefes)', fontsize=13, color=GOLD, fontweight='bold')
ax.set_xlabel('Score Promedio de Efectividad', fontsize=11)
ax.set_xlim(0, 105)
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('notebooks/output/ranking_armas.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 4. Análisis Streaming — Resultados del Job de Flink

In [ ]:
# Rankings en tiempo real (resultado de ventanas Flink)
rankings_df = pd.read_sql("""
    SELECT w.name AS weapon, b.name AS boss,
           r.win_rate, r.total_attempts, r.total_victories,
           r.avg_time_to_kill, r.updated_at
    FROM real_time_rankings r
    JOIN weapons w ON r.weapon_id = w.weapon_id
    JOIN bosses  b ON r.boss_id   = b.boss_id
    ORDER BY r.win_rate DESC
""", conn)

if rankings_df.empty:
    print('⚠️ No hay datos de streaming todavía. Ejecuta el demo: make demo')
else:
    print(f'Rankings disponibles: {len(rankings_df)} combinaciones')
    display(rankings_df.head(10))

In [ ]:
if not rankings_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Win rate vs intentos
    ax = axes[0]
    scatter = ax.scatter(
        rankings_df['total_attempts'], rankings_df['win_rate'],
        c=rankings_df['avg_time_to_kill'], cmap='plasma',
        s=rankings_df['total_victories'].clip(10) * 3,
        alpha=0.8, edgecolors='none'
    )
    plt.colorbar(scatter, ax=ax, label='TTK promedio (s)')
    ax.set_xlabel('Total de Intentos', fontsize=11)
    ax.set_ylabel('Win Rate (%)', fontsize=11)
    ax.set_title('⚡ Win Rate vs Popularidad (Flink Stream)', fontsize=12, color=GOLD, fontweight='bold')
    ax.axhline(y=50, color='#ff7b72', linestyle='--', alpha=0.5, label='50% WR')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # Distribución win rates
    ax2 = axes[1]
    ax2.hist(rankings_df['win_rate'], bins=15, color=GOLD, edgecolor='#0d1117', alpha=0.9)
    ax2.axvline(rankings_df['win_rate'].mean(), color='#58a6ff', linestyle='--', 
                label=f'Media: {rankings_df["win_rate"].mean():.1f}%')
    ax2.set_xlabel('Win Rate (%)', fontsize=11)
    ax2.set_ylabel('Frecuencia', fontsize=11)
    ax2.set_title('📊 Distribución de Win Rates', fontsize=12, color=GOLD, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('notebooks/output/streaming_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.show()

## 5. Detección de Meta-Shifts

In [ ]:
shifts_df = pd.read_sql("""
    SELECT w.name AS weapon, b.name AS boss,
           m.shift_type, m.previous_popularity AS prev_wr,
           m.current_popularity AS curr_wr,
           m.change_percentage, m.detected_at
    FROM meta_shifts m
    JOIN weapons w ON m.weapon_id = w.weapon_id
    LEFT JOIN bosses b ON m.boss_id = b.boss_id
    ORDER BY m.detected_at DESC
""", conn)

if shifts_df.empty:
    print('⚠️ Ningún Meta-Shift detectado aún (se necesitan >15% de variación entre ventanas)')
    print('   Ejecuta: make demo  y espera ~10 minutos de procesamiento Flink')
else:
    print(f'🚨 {len(shifts_df)} Meta-Shifts detectados:')
    display(shifts_df)

    # Visualización de variaciones
    fig, ax = plt.subplots(figsize=(10, 5))
    colors_map = {'emerging': '#3fb950', 'declining': '#f85149', 'dominant': '#d29922'}
    for _, row in shifts_df.iterrows():
        color = colors_map.get(row['shift_type'], '#8b949e')
        label = f"{row['weapon']} vs {row['boss']}"
        ax.barh(label, row['change_percentage'], color=color, alpha=0.8, edgecolor='none')
    
    ax.axvline(x=15, color='white', linestyle='--', alpha=0.5, linewidth=1, label='Umbral 15%')
    ax.set_xlabel('Variación en Win Rate (%)', fontsize=11)
    ax.set_title('🚨 Meta-Shifts Detectados por Flink', fontsize=13, color=GOLD, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(axis='x', alpha=0.3)
    
    patches = [mpatches.Patch(color=v, label=k.capitalize()) for k, v in colors_map.items()]
    ax.legend(handles=patches, loc='lower right')
    
    plt.tight_layout()
    plt.savefig('notebooks/output/meta_shifts.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.show()

## 6. Conclusiones

| Componente | Hallazgo Clave |
|---|---|
| **Spark Batch** | Blasphemous Blade y Dark Moon Greatsword son las armas con mayor efectividad teórica |
| **Flink Stream** | El metagame real diverge del teórico: Rivers of Blood lidera vs Malenia pese a no ser la arma con mejor score estático |
| **Meta-Shift** | El sistema detecta cambios de tendencia en ventanas de 5 minutos con umbral configurable (15%) |
| **Latencia E2E** | Desde que un jugador vence un jefe hasta que aparece en el dashboard: ~30 segundos |

### Arquitectura Validada
```
Simulador → Kafka → Flink (5min windows) → PostgreSQL → Streamlit/Grafana
                                         ↓
                               meta-shift-alerts → Discord Bot
```

> **Gabriel Cardona Medina** — Big Data Aplicado 2026

In [ ]:
conn.close()
print('✅ Análisis completado. Conexión cerrada.')